# JobFit — Phase 5 : Chatbot RAG

Base vectorielle ChromaDB + chatbot de conseils candidature.

In [1]:
import sys, os

base = os.path.expanduser('~/WORKSPACE /PROJETS FIN FORMATION JEDHA/NOUVEAU TAFF/jobfit_phase2')
sys.path.insert(0, os.path.join(base, 'src/rag'))
sys.path.insert(0, os.path.join(base, 'src/ml'))
sys.path.insert(0, os.path.join(base, 'src/nlp'))
sys.path.insert(0, os.path.join(base, 'src/api'))

os.environ['DATA_PROCESSED'] = os.path.join(base, 'data/processed')
os.environ['CHROMA_DIR']     = os.path.join(base, 'data/chroma')

import sqlite3
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from cv_parser import parse_cv
from embeddings import EmbeddingEngine
from chatbot import build_vector_store, load_vector_store, JobFitChatbot

DATA_DIR   = os.path.join(base, 'data/processed')
CHROMA_DIR = os.path.join(base, 'data/chroma')
CV_PATH    = os.path.join(base, 'data/cv/sample_cv.pdf')

os.makedirs(CHROMA_DIR, exist_ok=True)
print('Setup OK.')

Setup OK.


## 1. Chargement CV + offres

In [2]:
cv_data = parse_cv(CV_PATH)
engine  = EmbeddingEngine()
cv_embedding = engine.embed_text(cv_data['enriched_text'])

conn      = sqlite3.connect(f'{DATA_DIR}/jobfit.db')
df_offres = pd.read_sql('SELECT * FROM offres', conn)
conn.close()

offres_embeddings = np.load(f'{DATA_DIR}/offres_embeddings.npy')
df_scored         = pd.read_csv(f'{DATA_DIR}/offres_ml_scored.csv')

print(f'CV : {cv_data["nb_competences"]} compétences')
print(f'Offres : {len(df_offres)}')

Parsing du CV : /Users/martialbayom/WORKSPACE /PROJETS FIN FORMATION JEDHA/NOUVEAU TAFF/jobfit_phase2/data/cv/sample_cv.pdf
  Titre          : DATA SCIENTIST - MACHINE LEARNING ENGINEER
  Compétences    : 47 trouvées
  Expérience     : 0 ans
  Formations     : 4
Chargement du modèle : paraphrase-multilingual-MiniLM-L12-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9051.60it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modèle chargé.
CV : 47 compétences
Offres : 562


## 2. Construction de la base vectorielle ChromaDB

In [3]:
collection = build_vector_store(df_offres, offres_embeddings)
print(f'Collection : {collection.count()} offres indexées')

Indexation de 562 offres dans ChromaDB...
Base vectorielle construite : 562 offres indexées.
Collection : 562 offres indexées


## 3. Recherche sémantique dans ChromaDB

In [4]:
from chatbot import search_similar_offres

results = search_similar_offres(collection, cv_embedding, n_results=5)

print('Top 5 offres (recherche vectorielle ChromaDB) :')
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f'\n{i+1}. {meta["intitule"]}')
    print(f'   {meta["entreprise"]} | {meta["lieu"]} | {meta["type_contrat"]}')
    print(f'   Compétences : {meta["competences"][:80]}...')

Top 5 offres (recherche vectorielle ChromaDB) :

1. ALTERNANCE - Data Scientist H/F
    | 92 - Boulogne-Billancourt | CDD
   Compétences : ...

2. ALTERNANCE - Data Scientist H/F
    | 92 - Boulogne-Billancourt | CDD
   Compétences : ...

3. ALTERNANCE - Data Scientist H/F
    | 92 - Boulogne-Billancourt | CDD
   Compétences : ...

4. ALTERNANCE - Data Scientist H/F
    | 92 - Boulogne-Billancourt | CDD
   Compétences : ...

5. ALTERNANCE - Data Scientist H/F
    | 92 - Boulogne-Billancourt | CDD
   Compétences : ...


## 4. Chatbot — analyse d'une offre spécifique

In [5]:
chatbot = JobFitChatbot(cv_data)

# Prendre la meilleure offre scorée
best_offre = df_scored.iloc[0].to_dict()
best_score = best_offre.get('ml_score_pct', 0)

print('=== ANALYSE DE LA MEILLEURE OFFRE ===')
print(f'Offre : {best_offre.get("intitule", "")}')
print(f'Score : {best_score}%')
print()
advice = chatbot.analyze_offre(best_offre, best_score)
print(advice)

=== ANALYSE DE LA MEILLEURE OFFRE ===
Offre : ALTERNANCE - Data Scientist H/F
Score : 99.9%

Analyse de ta candidature pour : ALTERNANCE - Data Scientist H/F
Entreprise : nan
Score de compatibilité : 99.9%

Excellent match ! Ton profil correspond bien à cette offre.

Compétences à mettre en avant ou à acquérir :
  • snowflake
  • azure

Conseils pour ta lettre de motivation :
  • Mets en avant : r, deep learning, databricks
  • Cite 1-2 projets concrets en lien avec le poste
  • Quantifie tes résultats (ex: amélioration de X%, traitement de Y données)
  • Souligne ta disponibilité rapide et ta flexibilité

Prochaine étape :
  → Personnalise ton CV pour faire ressortir les mots-clés de l'offre
  → Recherche nan sur LinkedIn avant l'entretien


## 5. Simulation conversation chatbot

In [6]:
questions = [
    'Quelles compétences me manquent pour ce poste ?',
    'Comment préparer ma lettre de motivation ?',
    'Comment me préparer pour l\'entretien ?',
    'Quel est mon score de compatibilité ?',
]

print('=== SIMULATION CONVERSATION CHATBOT ===\n')
for q in questions:
    print(f'Vous : {q}')
    response = chatbot.chat(q, offre=best_offre, score=best_score)
    print(f'JobFit : {response}')
    print('-' * 50)

=== SIMULATION CONVERSATION CHATBOT ===

Vous : Quelles compétences me manquent pour ce poste ?
JobFit : Compétences manquantes pour cette offre :
  • snowflake
  • azure
--------------------------------------------------
Vous : Comment préparer ma lettre de motivation ?
JobFit : Pour ta lettre de motivation :
  1. Ouvre avec ce qui t'attire spécifiquement chez nan
  2. Cite 2 projets de ton CV en lien direct avec le poste
  3. Montre ta compréhension de leurs enjeux data
  4. Termine sur ta disponibilité et motivation
--------------------------------------------------
Vous : Comment me préparer pour l'entretien ?
JobFit : Pour préparer l'entretien :
  • Revois les fondamentaux ML/stats (algo, complexité)
  • Prépare 3 projets à présenter (contexte, méthode, résultats)
  • Prépare des questions sur leur stack data
  • Entraîne-toi sur LeetCode (SQL + Python)
--------------------------------------------------
Vous : Quel est mon score de compatibilité ?
JobFit : Ton score de compatibili

## 6. Test mode copier-coller (offre externe)

In [7]:
# Simuler une offre copiée-collée depuis LinkedIn/Indeed
offre_externe = {
    'intitule':    'Data Scientist - NLP',
    'entreprise':  'BNP Paribas',
    'lieu':        'Paris (75)',
    'type_contrat': 'CDI',
    'description': '''
        Nous recherchons un Data Scientist spécialisé NLP pour rejoindre notre équipe.
        Missions : développement de modèles de traitement du langage naturel,
        fine-tuning de modèles BERT/CamemBERT, déploiement via FastAPI.
        Stack : Python, PyTorch, Transformers, Docker, AWS, MLflow.
        Profil : 2+ ans d\'expérience, connaissance des LLMs, RAG appréciée.
    ''',
    'competences': 'Python, NLP, BERT, PyTorch, Transformers, Docker, AWS, MLflow, FastAPI',
}

# Scorer cette offre externe
offre_embedding = engine.embed_text(
    f"{offre_externe['intitule']} {offre_externe['description']} {offre_externe['competences']}"
)
score_externe = float(np.dot(cv_embedding, offre_embedding)) * 100

print(f'Offre externe : {offre_externe["intitule"]}')
print(f'Score : {score_externe:.1f}%')
print()
print(chatbot.analyze_offre(offre_externe, score_externe))

Offre externe : Data Scientist - NLP
Score : 73.4%

Analyse de ta candidature pour : Data Scientist - NLP
Entreprise : BNP Paribas
Score de compatibilité : 73.4%

Excellent match ! Ton profil correspond bien à cette offre.

Compétences à mettre en avant ou à acquérir :
  • camembert
  • rag

Conseils pour ta lettre de motivation :
  • Mets en avant : python, r, nlp, transformers
  • Cite 1-2 projets concrets en lien avec le poste
  • Quantifie tes résultats (ex: amélioration de X%, traitement de Y données)
  • Exprime ta motivation pour un engagement long terme

Prochaine étape :
  → Personnalise ton CV pour faire ressortir les mots-clés de l'offre
  → Recherche BNP Paribas sur LinkedIn avant l'entretien
